# Project 3: Reading the Chinook Database into pandas

**Sebastian Perdomo | IS 362**

- **RDBMS:** SQLite. The whole database is a single file, so there's no server to install.
- **Middleware:** SQLAlchemy. For SQLite it uses Python's built-in `sqlite3` driver, so no extra driver is needed.
- **Connection string:** `sqlite:///Chinook_Sqlite.sqlite`, which points to the database file in the same folder as this notebook.

## 1. Set up

The database is the SQLite version of Chinook from the project's GitHub releases. The file `Chinook_Sqlite.sqlite`
sits in the same folder as this notebook.

In [1]:
import pandas as pd
import sqlalchemy as sqla

DB_FILE = "Chinook_Sqlite.sqlite"

print("SQLAlchemy version:", sqla.__version__)

SQLAlchemy version: 2.0.51


## 2. Connect

In [2]:
db = sqla.create_engine(f"sqlite:///{DB_FILE}")

sqla.inspect(db).get_table_names()

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

## 3. Join the five tables

Customers connect to the tracks they bought through their invoices:

Customer → Invoice → InvoiceLine → Track → Album

Each line of an invoice is one purchased track, and each track belongs to one album.

In [3]:
sql = '''SELECT c.LastName, c.FirstName, t.Name, a.Title
         FROM Customer c
         JOIN Invoice i      ON c.CustomerId = i.CustomerId
         JOIN InvoiceLine il ON i.InvoiceId  = il.InvoiceId
         JOIN Track t        ON il.TrackId   = t.TrackId
         JOIN Album a        ON t.AlbumId    = a.AlbumId
         ORDER BY c.LastName, c.FirstName'''

purchases = pd.read_sql(sql, db)
purchases.head()

,LastName,FirstName,Name,Title
0,Almeida,Roberto,Right Next Door to Hell,Use Your Illusion I
1,Almeida,Roberto,In The Evening,In Through The Out Door
2,Almeida,Roberto,Fool In The Rain,In Through The Out Door
3,Almeida,Roberto,Saudade Dos Aviões Da Panair (Conversando No Bar),Minas
4,Almeida,Roberto,Caso Você Queira Saber,Minas


## 4. Check the result

Every invoice line is one track purchase, so the DataFrame should have exactly one
row per invoice line.

In [4]:
invoice_lines = pd.read_sql("SELECT COUNT(*) AS n FROM InvoiceLine", db)["n"][0]

print("Rows in DataFrame:", len(purchases))
print("Invoice lines:    ", invoice_lines)
print("Customers:        ", purchases[["LastName", "FirstName"]].drop_duplicates().shape[0])

Rows in DataFrame: 2240
Invoice lines:     2240
Customers:         59
